# Test Query Team Execution

This notebook demonstrates two ways to run the query team workflow:
1. **Using QueryManager**: Simulates the standard way of submitting a query and getting the final result.
2. **Using Direct Graph Stream**: Directly interacts with the LangGraph instance to observe intermediate steps.

## Part 0: Setup (Imports and Ontology)

In [1]:
import sys
import os
sys.path.append(r"D:\\CursorProj\\Chem-Ontology-Constructor")
os.environ["PROJECT_ROOT"] = "D:\\\\CursorProj\\\\Chem-Ontology-Constructor\\\\"

from owlready2 import get_ontology
# 从 config.settings 导入 ONTOLOGY_SETTINGS 而不是 ONTOLOGY_CONFIG
from config.settings import ONTOLOGY_SETTINGS
# 本体现在在 ONTOLOGY_SETTINGS 初始化时加载，如果需要，可以通过 ONTOLOGY_SETTINGS.ontology 访问
# 例如: onto = ONTOLOGY_SETTINGS.ontology
# onto_additional = get_ontology("data/ontology/test.owl").load() # 可选的第二个本体

Setting owlready2.JAVA_EXE globally from settings.yaml: C:\Program Files\Java\jdk-23\bin\java.exe


In [2]:
# Required Imports
import sys
import os
import json
import time
from typing import Dict, Any, List
from owlready2 import *
import asyncio # Needed for owlready2 async operations in some envs

# Import the OntologySettings class
from config.settings import OntologySettings, ONTOLOGY_SETTINGS # Keep ONTOLOGY_SETTINGS import for potential base_iri access

# Import necessary LLM and Query Team components
try:
    from autology_constructor.idea.query_team import QueryManager, Query, QueryStatus, create_query_graph
    from autology_constructor.idea.query_team.ontology_tools import OntologyTools
    from autology_constructor.idea.common.llm_provider import get_cached_default_llm
    print("Modules imported successfully.")
except ModuleNotFoundError as e:
    print(f"Error importing modules: {e}")
    print(f"Current sys.path: {sys.path}")

# Ensure LLM Provider is configured
try:
    llm = get_cached_default_llm()
    print("LLM Provider initialized successfully.")
except Exception as e:
    print(f"Error initializing LLM Provider: {e}\nPlease ensure API keys or necessary configurations are set.")
    llm = None

# --- Ontology Setup ---
print("Setting up test ontology using a new OntologySettings instance...")

# Define parameters for the new OntologySettings instance
# Assuming 'backup-2.owl' and 'backup-2-closed.owl' exist in 'data/ontology'
# Use the project root defined in the previous cell
project_root_path = os.environ.get("PROJECT_ROOT", ".")
ontology_dir = os.path.join(project_root_path, "data", "ontology")
# You might want to use the base_iri from the default settings or define a specific one for testing
test_base_iri = ONTOLOGY_SETTINGS.base_iri if 'ONTOLOGY_SETTINGS' in locals() else "http://www.test.org/chem_ontologies/backup-2"

try:
    # Instantiate OntologySettings directly
    test_ontology_settings = OntologySettings(
        base_iri=test_base_iri,
        ontology_file_name="backup-2.owl",  # Use the desired ontology file
        directory_path=ontology_dir,
        closed_ontology_file_name="backup-2-closed.owl" # Adjust if your closed file has a different name pattern
    )
    # Access the loaded ontology via the instance's property
    test_onto = test_ontology_settings.ontology
    print(f"Successfully loaded ontology: {test_onto.base_iri}")
    print(f"From file: {test_ontology_settings.ontology_file_name} in {test_ontology_settings.directory_path}")

    # Optional: Print some details about the loaded ontology
    # print(f"Test Ontology '{test_onto.base_iri}' loaded with:")
    # print(f"- Classes ({len(list(test_onto.classes()))}): {[c.name for c in list(test_onto.classes())[:5]]}...") # Print first 5
    # print(f"- Individuals ({len(list(test_onto.individuals()))}): {[i.name for i in list(test_onto.individuals())[:5]]}...")
    # print(f"- Object Properties ({len(list(test_onto.object_properties()))}): {[p.name for p in list(test_onto.object_properties())[:5]]}...")
    # print(f"- Data Properties ({len(list(test_onto.data_properties()))}): {[p.name for p in list(test_onto.data_properties())[:5]]}...")

except Exception as e:
    print(f"Error creating OntologySettings or loading ontology 'backup-2.owl': {e}")
    print(f"Please ensure 'backup-2.owl' exists in '{ontology_dir}' and settings are correct.")
    test_onto = None # Set to None if loading failed

# # --- Old way (commented out) ---
# # print("Creating a simple in-memory ontology...")
# # # It's good practice to clear existing ontologies from the default world if running cells repeatedly
# # for o in list(default_world.ontologies.values()):
# #     if callable(getattr(o, '__destroy__', None)):
# #         try:
# #             destroy_entity(o)
# #         except Exception as destroy_err:
# #             print(f"Error destroying {o.base_iri}: {destroy_err}")
# #     else:
# #         print(f"Skipping destroy for non-callable __destroy__ or missing: {o.base_iri}")
# # test_onto_old = get_ontology("data/ontology/test.owl").load()
# # for o in list(default_world.ontologies.values()):
# #     print(f"has：{o.base_iri}")
# # print(f"Test Ontology '{test_onto_old.base_iri}' created with:\\n- Classes: {[c.name for c in test_onto_old.classes()]}\\n- Individuals: {[i.name for i in test_onto_old.individuals()]}\\n- Object Properties: {[p.name for p in test_onto_old.object_properties()]}\\n- Data Properties: {[p.name for p in test_onto_old.data_properties()]}\")

# Run async tasks if needed by owlready2 backend (usually not necessary for simple loading)
# try:
#     loop = asyncio.get_event_loop()
# except RuntimeError:
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)
# loop.run_until_complete(asyncio.sleep(0)) # Run pending async tasks

Modules imported successfully.
LLM Provider initialized successfully.
Setting up test ontology using a new OntologySettings instance...
Successfully loaded ontology: http://www.test.org/chem_ontologies/chem_ontology.owl#
From file: backup-2.owl in D:\\CursorProj\\Chem-Ontology-Constructor\\data\ontology


## Part 1: Execution via QueryManager


**QA Pair 1**

* **Question:** What description is provided for calix[4]pyrrole in the ontology?
* **Answer:** Calix[4]pyrroles are macrocyclic compounds typically synthesized from the acid-catalyzed condensation of pyrrole and ketones. They feature a central cavity rich in electrons due to the pyrrole rings.
* **Difficulty:** Level 1 (Specific Fact/Entity Lookup)

**QA Pair 2**

* **Question:** What is the pKa value mentioned for methanesulfonic acid (MSA)?
* **Answer:** The ontology mentions the pKa value of methanesulfonic acid (MSA) is -1.9.
* **Difficulty:** Level 1 (Specific Fact/Entity Lookup)

**QA Pair 3**

* **Question:** What type of reaction is used to synthesize aryl-extended calix(4)pyrroles (AE-C(4)Ps)?
* **Answer:** Aryl-extended calix(4)pyrroles (AE-C(4)Ps) are synthesized via a cyclocondensation reaction.
* **Difficulty:** Level 2 (Simple Relationship / Instance Enumeration)

**QA Pair 4**

* **Question:** List some specific types (subclasses) of macrocyclic receptors mentioned in the ontology.
* **Answer:** Some types include calix(4)pyrrole, calix(4)arene, cryptand, oligopyrrolic cage, and cucurbit(n)uril.
* **Difficulty:** Level 2 (Simple Relationship / Instance Enumeration)

**QA Pair 5**

* **Question:** According to the ontology, what can enhance the yield and selectivity during the synthesis of aryl-extended calix(4)pyrroles (AE-C(4)Ps)?
* **Answer:** The use of templates is mentioned to enhance the yield and selectivity in the synthesis of AE-C(4)Ps.
* **Difficulty:** Level 3 (Multi-Step Information Chaining)

**QA Pair 6**

* **Question:** How do the aromatic cavities of calix[4]pyrrole and calix[4]arene differ according to the ontology's description?
* **Answer:** The ontology notes that the calix[4]pyrrole cavity is defined by electropositive C–H bonds pointing inwards, whereas the calix[4]arene cavity is lined by electron-rich π surfaces.
* **Difficulty:** Level 3 (Comparative Queries / Multi-Step Information Chaining)

**QA Pair 7**

* **Question:** Identify specific calix[4]pyrrole derivatives mentioned in the ontology that utilize anion-pi interactions for anion binding.
* **Answer:** Aryl-extended calix(4)pyrroles (AE-C(4)Ps) and Super aryl-extended calix(4)pyrroles (SAE-C(4)Ps) are mentioned as utilizing anion-pi interactions for binding anions.
* **Difficulty:** Level 4 (Complex Aggregation / Conditional Filtering)

**QA Pair 8**

* **Question:** Summarize the different applications or functions mentioned for various oligopyrrolic cages within the ontology.
* **Answer:** Oligopyrrolic cages are described as being used for gas absorption, anion recognition/binding, catalysis, and binding specific guests like fullerenes (e.g., C60 by dimeric capsules).
* **Difficulty:** Level 4 (Complex Aggregation / Conditional Filtering)

**QA Pair 9**

* **Question:** According to the ontology, why is dynamic covalent chemistry (DCC) a useful strategy for constructing oligopyrrolic cages?
* **Answer:** Dynamic covalent chemistry is highlighted because its reversible nature allows for "error correction" during assembly, enabling the formation of complex, thermodynamically stable cage structures that might be difficult to obtain via irreversible reactions.
* **Difficulty:** Level 5 (Abstract Reasoning / Explanation / Synthesis)

**QA Pair 10**

* **Question:** Based on the ontology, how do structural modifications like aryl extensions (forming SAE-C(4)Ps) or adopting specific conformations (like four-wall αααα-AE-C(4)Ps) influence the anion binding properties of calix[4]pyrrole derivatives?
* **Answer:** The ontology suggests that extending the aryl groups (as in SAE-C(4)Ps) creates a deeper aromatic cavity, leading to stronger anion binding compared to AE-C(4)Ps, often facilitated by anion-pi interactions. Adopting specific conformations like the four-wall αααα isomer in AE-C(4)Ps also results in effective anion receptors. These modifications enhance the structural features responsible for guest complexation.
* **Difficulty:** Level 5 (Abstract Reasoning / Explanation / Synthesis)


In [3]:
qas = [
  {
    "question": "What description is provided for calix[4]pyrrole in the ontology?",
    "answer": "Calix[4]pyrroles are macrocyclic compounds typically synthesized from the acid-catalyzed condensation of pyrrole and ketones. They feature a central cavity rich in electrons due to the pyrrole rings.",
    "difficulty": 1
  },
  {
    "question": "What is the pKa value mentioned for methanesulfonic acid (MSA)?",
    "answer": "The ontology mentions the pKa value of methanesulfonic acid (MSA) is -1.9.",
    "difficulty": 1
  },
  {
    "question": "What type of reaction is used to synthesize aryl-extended calix(4)pyrroles (AE-C(4)Ps)?",
    "answer": "Aryl-extended calix(4)pyrroles (AE-C(4)Ps) are synthesized via a cyclocondensation reaction.",
    "difficulty": 2
  },
  {
    "question": "List some specific types (subclasses) of macrocyclic receptors mentioned in the ontology.",
    "answer": "Some types include calix(4)pyrrole, calix(4)arene, cryptand, oligopyrrolic cage, and cucurbit(n)uril.",
    "difficulty": 2
  },
  {
    "question": "According to the ontology, what can enhance the yield and selectivity during the synthesis of aryl-extended calix(4)pyrroles (AE-C(4)Ps)?",
    "answer": "The use of templates is mentioned to enhance the yield and selectivity in the synthesis of AE-C(4)Ps.",
    "difficulty": 3
  },
  {
    "question": "How do the aromatic cavities of calix[4]pyrrole and calix[4]arene differ according to the ontology's description?",
    "answer": "The ontology notes that the calix[4]pyrrole cavity is defined by electropositive C–H bonds pointing inwards, whereas the calix[4]arene cavity is lined by electron-rich π surfaces.",
    "difficulty": 3
  },
  {
    "question": "Identify specific calix[4]pyrrole derivatives mentioned in the ontology that utilize anion-pi interactions for anion binding.",
    "answer": "Aryl-extended calix(4)pyrroles (AE-C(4)Ps) and Super aryl-extended calix(4)pyrroles (SAE-C(4)Ps) are mentioned as utilizing anion-pi interactions for binding anions.",
    "difficulty": 4
  },
  {
    "question": "Summarize the different applications or functions mentioned for various oligopyrrolic cages within the ontology.",
    "answer": "Oligopyrrolic cages are described as being used for gas absorption, anion recognition/binding, catalysis, and binding specific guests like fullerenes (e.g., C60 by dimeric capsules).",
    "difficulty": 4
  },
  {
    "question": "According to the ontology, why is dynamic covalent chemistry (DCC) a useful strategy for constructing oligopyrrolic cages?",
    "answer": "Dynamic covalent chemistry is highlighted because its reversible nature allows for \"error correction\" during assembly, enabling the formation of complex, thermodynamically stable cage structures that might be difficult to obtain via irreversible reactions.",
    "difficulty": 5
  },
  {
    "question": "Based on the ontology, how do structural modifications like aryl extensions (forming SAE-C(4)Ps) or adopting specific conformations (like four-wall αααα-AE-C(4)Ps) influence the anion binding properties of calix[4]pyrrole derivatives?",
    "answer": "The ontology suggests that extending the aryl groups (as in SAE-C(4)Ps) creates a deeper aromatic cavity, leading to stronger anion binding compared to AE-C(4)Ps, often facilitated by anion-pi interactions. Adopting specific conformations like the four-wall αααα isomer in AE-C(4)Ps also results in effective anion receptors. These modifications enhance the structural features responsible for guest complexation.",
    "difficulty": 5
  }
]

In [4]:
revised_qas = [
  {
    "question": "Can you provide a description for calix[4]pyrrole?",
    "answer": "Calix[4]pyrroles are macrocyclic compounds typically synthesized from the acid-catalyzed condensation of pyrrole and ketones. They feature a central cavity rich in electrons due to the pyrrole rings.",
    "difficulty": 1
  },
  {
    "question": "What is the pKa value of methanesulfonic acid (MSA)?",
    "answer": "The pKa value of methanesulfonic acid (MSA) is -1.9.",
    "difficulty": 1
  },
  {
    "question": "What type of reaction is used to synthesize aryl-extended calix(4)pyrroles (AE-C(4)Ps)?",
    "answer": "Aryl-extended calix(4)pyrroles (AE-C(4)Ps) are synthesized via a cyclocondensation reaction.",
    "difficulty": 2
  },
  {
    "question": "Can you list some specific types of macrocyclic receptors?",
    "answer": "Some types include calix(4)pyrrole, calix(4)arene, cryptand, oligopyrrolic cage, and cucurbit(n)uril.",
    "difficulty": 2
  },
  {
    "question": "How can the yield and selectivity be enhanced during the synthesis of aryl-extended calix(4)pyrroles (AE-C(4)Ps)?",
    "answer": "The use of templates can enhance the yield and selectivity in the synthesis of AE-C(4)Ps.",
    "difficulty": 3
  },
  {
    "question": "How do the aromatic cavities of calix[4]pyrrole and calix[4]arene differ?",
    "answer": "The calix[4]pyrrole cavity is defined by electropositive C–H bonds pointing inwards, whereas the calix[4]arene cavity is lined by electron-rich π surfaces.",
    "difficulty": 3
  },
  {
    "question": "Which specific calix[4]pyrrole derivatives utilize anion-pi interactions for anion binding?",
    "answer": "Aryl-extended calix(4)pyrroles (AE-C(4)Ps) and Super aryl-extended calix(4)pyrroles (SAE-C(4)Ps) utilize anion-pi interactions for binding anions.",
    "difficulty": 4
  },
  {
    "question": "What are the different applications or functions of various oligopyrrolic cages?",
    "answer": "Oligopyrrolic cages are used for gas absorption, anion recognition/binding, catalysis, and binding specific guests like fullerenes (e.g., C60 by dimeric capsules).",
    "difficulty": 4
  },
  {
    "question": "Why is dynamic covalent chemistry (DCC) a useful strategy for constructing oligopyrrolic cages?",
    "answer": "Dynamic covalent chemistry's reversible nature allows for \"error correction\" during assembly, enabling the formation of complex, thermodynamically stable cage structures that might be difficult to obtain via irreversible reactions.",
    "difficulty": 5
  },
  {
    "question": "How do structural modifications like aryl extensions (forming SAE-C(4)Ps) or adopting specific conformations (like four-wall αααα-AE-C(4)Ps) influence the anion binding properties of calix[4]pyrrole derivatives?",
    "answer": "Extending the aryl groups (as in SAE-C(4)Ps) creates a deeper aromatic cavity, leading to stronger anion binding compared to AE-C(4)Ps, often facilitated by anion-pi interactions. Adopting specific conformations like the four-wall αααα isomer in AE-C(4)Ps also results in effective anion receptors. These modifications enhance the structural features responsible for guest complexation.",
    "difficulty": 5
  }
]

# You can optionally print it to verify
# import json
# print(json.dumps(revised_qas, indent=2))

In [5]:
num = 10
# 定义新的十个查询
queries = [item["question"] for item in qas[:num]]

revised_queries = [item["question"] for item in revised_qas[:num]]

# 为所有查询定义统一的上下文
query_context = {
    "ontology": test_ontology_settings,
    "originating_team": "test_notebook",
    "originating_stage": "manual_test",
    "query_type": "information_retrieval" # 对所有查询使用信息检索类型
}

print(len(queries),len(revised_queries))

10 10


In [6]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:
    print("--- Starting QueryManager Test ---")
    query_manager = QueryManager()

    # 更新类缓存
    print("Updating class name cache...")
    query_manager.update_class_name_cache(test_onto)
    # 打印部分缓存内容以确认
    if query_manager.class_name_cache:
         print(f"Cache content (first 10): {query_manager.class_name_cache[:10]}...")
    else:
         print("Class name cache is empty.")


    # 启动管理器
    print("Starting QueryManager...")
    query_manager.start()

    # 提交多个查询并收集futures
    futures = []
    print(f"\\nSubmitting {len(queries)} queries...")
    for i, query_text in enumerate(queries):
        print(f"Submitting query {i+1}: '{query_text[:80]}...'") # 打印部分查询文本
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        futures.append((i+1, query_text, future))

    print("\nAll queries submitted.")

--- Starting QueryManager Test ---
Updating class name cache...
Class name cache updated with 351 classes.
Cache content (first 10): ['1,3,5-triethyl-2,4,6-trimethylamine', '1,3-diynyl', '1,4-triazole', '1:1_inclusion_complexes', '1H_NMR_spectroscopic_titration', '1H_NMR_spectroscopic_titrations', '1H_NMR_spectrum', '1_palmitoyl_2_oleoyl_sn_glycero_3_phosphocholine(POPC)', '23a', '25⊂(24)2']...
Starting QueryManager...
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
\nSubmitting 1 queries...
Submitting query 1: 'What description is provided for calix[4]pyrrole in the ontology?...'

All queries submitted.
[('system', 'You are an expert ontology query parser. Your task is to convert natural language queries into a structured format.\n1. Strictly adhere to the NormalizedQuery JSON schema for the output.\n2. Refer to the provided list of available ontology classes to identify entities.\n3. Refer to the provided lists of data properties and object p

http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
{
  "results": [
    {
      "tool": "parse_class_definition",
      "params": {
        "class_names": [
          "calix(4)pyrrole"
        ]
      },
      "result": {
        "calix(4)pyrrole": {
          "basic_info": {
            "name": "calix(4)pyrrole",
            "information": [
              "A macrocycle containing four pyrrole rings connected through their pyrrolic 2- and 5-positions by tetra-substituted sp3 carbon atoms (meso-substituents).",
              "N-methyldiethanolamine is studied in methanol or methanol aqueous solutions as a solvent, with its concentration varying from 0 to 100% mass."
            ]
          },
          "properties": {
            "data": [],
            "object": []
          },
          "hierarchy": {
            "parents": [],
            "ch

In [7]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:    
    # 等待并获取所有结果
    print("Waiting for queries completion...")
    try:
        for i, query_text, future in futures:
            print(f"\nProcessing results for query {i}: '{query_text}'")
            # 等待合理的时间（根据需要调整）
            final_result_dict = future.result(timeout=120)
            print(f"Query {i} completed.")
            # 美观打印最终状态字典
            print(f"\n--- Final State Dictionary for Query {i} ---")
            # 使用default=str处理潜在的不可序列化对象，如本体引用
            print(json.dumps(final_result_dict, indent=2, default=str))
    except Exception as e:
        print(f"Error getting query result: {e}")
        if future.done() and future.exception():
             print(f"Future exception details: {future.exception()}")
    finally:
        # 停止管理器
        print("\nStopping QueryManager...")
        query_manager.stop()
        print("QueryManager stopped.")

    print("--- QueryManager Test Finished ---")


Waiting for queries completion...

Processing results for query 1: 'What description is provided for calix[4]pyrrole in the ontology?'
Query 1 completed.

--- Final State Dictionary for Query 1 ---
{
  "query": "What description is provided for calix[4]pyrrole in the ontology?",
  "source_ontology": "OntologySettings(base_iri='http://www.test.org/chem_ontologies/', ontology_file_name='backup-2.owl', directory_path='D:\\\\\\\\CursorProj\\\\\\\\Chem-Ontology-Constructor\\\\\\\\data\\\\ontology', closed_ontology_file_name='backup-2-closed.owl')",
  "query_type": "information_retrieval",
  "query_strategy": "tool_sequence",
  "originating_team": "test_notebook",
  "originating_stage": "manual_test",
  "available_classes": [
    "1,3,5-triethyl-2,4,6-trimethylamine",
    "1,3-diynyl",
    "1,4-triazole",
    "1:1_inclusion_complexes",
    "1H_NMR_spectroscopic_titration",
    "1H_NMR_spectroscopic_titrations",
    "1H_NMR_spectrum",
    "1_palmitoyl_2_oleoyl_sn_glycero_3_phosphocholine(POPC

In [6]:
# 定义回调函数处理Future结果并使用agent生成回答

def process_result_with_agent(result_dict, query_text):
    """
    Use an agent to process query results and generate a natural language response
    
    Args:
        result_dict: Query result dictionary
        query_text: Original query text
    
    Returns:
        str: Natural language response generated by the agent
    """
    # Extract query results information from the result
    if "query_results" in result_dict and "results" in result_dict["query_results"]:
        query_results = result_dict["query_results"]["results"]
    else:
        return f"I'm sorry, I couldn't find valid information about '{query_text}'."
    
    # Construct the prompt in English
    prompt = f"""
**Role:** You are an expert Chemistry Researcher.

**Task:** Provide a clear, accurate, and comprehensive answer to the user's question. You should leverage your own expert knowledge, **judiciously enhancing and verifying** it with **relevant and applicable information** selected from the 'Ontology query results'.

**User Question:**
{query_text}

**Information Source (Ontology Query Results for Enhancement & Verification):**
{query_results}

**Response Guidelines:**
* **Knowledge Integration:** Synthesize your broad chemical knowledge with **pertinent details** from the 'Information Source'.
* **Selective Use of Source:** Critically evaluate the 'Information Source'. **Incorporate specific details** (e.g., data points like pKa values, reaction types, precise definitions, specific examples) **only when they directly enhance the accuracy, specificity, or completeness of the answer to the user's question.** Do not feel obligated to include all provided information; prioritize relevance to the query.
* **Verification and Conflict:** Use the source to verify facts where appropriate. If there's a conflict between your general knowledge and the source, prioritize the source's specific data **if it is relevant to the question and appears accurate**, but use your expert judgment to omit information that seems erroneous or irrelevant to the user's query.
* **Synthesis:** Weave together your general knowledge and the selected source information into a coherent, well-structured response.
* **Clarity & Tone:** Use precise, professional chemical language. Aim for accessibility by briefly explaining potentially niche terms if needed.
* **Directness & Comprehensiveness:** Address all parts of the user's question directly and thoroughly, enriched by the appropriately selected information.
* **Source Attribution:** Do **not** mention "ontology" or refer to the 'Information Source' explicitly (e.g., avoid "according to the provided data..."). Present the integrated information as established chemical facts.

**Answer:**
"""
    
    # Generate response using LLM
    try:
        response = llm.invoke(prompt)
        return response
    except Exception as e:
        return f"Error generating response: {e}"

def query_result_callback(future, query_idx, query_text):
    """Callback function to process Future results"""
    try:
        print(f"\nProcessing callback for query {query_idx}: '{query_text}'")
        
        # Get the future result
        result_dict = future.result(timeout=5)  # Small timeout to avoid indefinite waiting
        
        # Process the result using the agent
        answer = process_result_with_agent(result_dict, query_text)
        
        # Print the agent-generated answer
        print(f"\n--- Agent Answer for Query {query_idx} ---")
        print(answer)
        print("------------------------------")
        print(answer.content)
        
        return answer
    except Exception as e:
        print(f"Error processing result in callback: {e}")
        if future.exception():
            print(f"Future exception details: {future.exception()}")
        return None

# Test code using callback functions to process query results

if not llm:
    print("Skipping callback test due to LLM initialization failure.")
else:
    print("\n--- Starting Callback Function Test ---")
    
    # Re-create query manager if needed
    if 'query_manager' not in locals() or not hasattr(query_manager, 'is_running') or not query_manager.is_running():
        query_manager = QueryManager()
        query_manager.update_all_caches(test_onto)
        query_manager.start()
    
    # 创建一个闭包函数来捕获回调返回的answer
    def create_answer_collector():
        # 在闭包中创建一个存储结果的字典
        answers = {}
        
        # 创建一个能捕获answer的回调函数
        def answer_collector(future, query_idx, query_text):
            try:
                result_dict = future.result(timeout=5)
                # 处理结果并获取answer
                answer = process_result_with_agent(result_dict, query_text)
                # 将answer存储在闭包的answers字典中
                answers[query_idx] = answer
                print(f"查询 {query_idx} 的答案已保存")
                return answer
            except Exception as e:
                print(f"处理结果时出错: {e}")
                return None
        
        # 返回回调函数和结果字典
        return answer_collector, answers

    # 创建回调函数和结果存储字典
    callback_collector, answers = create_answer_collector()

    # 提交查询并注册回调
    callback_futures = []
    for i, query_text in enumerate(queries):
        question = revised_queries[i]
        print(f"提交查询 {i+1}: '{query_text}'")
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        
        # 使用functools.partial创建带参数的回调函数
        from functools import partial
        callback_func = partial(callback_collector, query_idx=i+1, query_text=question)
        
        # 注册回调函数
        future.add_done_callback(callback_func)
        callback_futures.append((i+1, query_text, future))
    
    # Wait for all Futures to complete (optional but ensures all callbacks execute)
    import concurrent.futures
    import time
    
    # Non-blocking check
    all_done = False
    wait_time = 0
    max_wait_time = 120  # Maximum wait time
    check_interval = 5  # Check interval
    
    print("\nWaiting for callbacks to execute...")
    while not all_done and wait_time < max_wait_time:
        all_done = all(future[2].done() for future in callback_futures)
        if not all_done:
            print(f"Waited {wait_time} seconds, continuing to wait for callbacks...")
            time.sleep(check_interval)
            wait_time += check_interval
    
    if all_done:
        print("\nAll callbacks have completed!")
    else:
        print(f"\nTimeout waiting, some queries may not have completed. Waited {wait_time} seconds.")
    
    # Stop query manager
    print("\nStopping QueryManager...")
    query_manager.stop()
    print("QueryManager stopped.")
    
    print("--- Callback Function Test Finished ---")


--- Starting Callback Function Test ---
Class name cache updated with 351 classes.
数据属性缓存更新完成，共 93 个属性
对象属性缓存更新完成，共 115 个属性
所有本体缓存更新完成
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
提交查询 1: 'What description is provided for calix[4]pyrrole in the ontology?'
提交查询 2: 'What is the pKa value mentioned for methanesulfonic acid (MSA)?'
提交查询 3: 'What type of reaction is used to synthesize aryl-extended calix(4)pyrroles (AE-C(4)Ps)?'
提交查询 4: 'List some specific types (subclasses) of macrocyclic receptors mentioned in the ontology.'
提交查询 5: 'According to the ontology, what can enhance the yield and selectivity during the synthesis of aryl-extended calix(4)pyrroles (AE-C(4)Ps)?'
提交查询 6: 'How do the aromatic cavities of calix[4]pyrrole and calix[4]arene differ according to the ontology's description?'
提交查询 7: 'Identify specific calix[4]pyrrole derivatives mentioned in the ontology that utilize anion-pi interactions for anion binding.'
提交查询 8: 'Summarize

In [7]:
for idx, answer in answers.items():
        print(f"查询 {idx} 的最终答案: {answer.content}")

查询 2 的最终答案: Methanesulfonic acid (MSA) is a strong organic acid commonly used in various chemical applications, including as a catalyst and in the synthesis of pharmaceuticals. The pKa value of methanesulfonic acid is approximately -1.92. This value indicates that MSA is a strong acid, as it has a negative pKa, which is characteristic of acids that dissociate almost completely in aqueous solutions. The low pKa value reflects its high acidity compared to many other organic acids, making it effective in protonating weak bases and facilitating various chemical reactions.
查询 1 的最终答案: Calix[4]pyrrole is a type of macrocyclic compound that consists of four pyrrole rings. These pyrrole units are connected through their 2- and 5-positions by sp³ hybridized carbon atoms, which are typically substituted with various groups, known as meso-substituents. This structure forms a cyclic arrangement that can encapsulate guest molecules, making calix[4]pyrrole an interesting subject in supramolecular ch

In [10]:
res_list = []
for i, ques in enumerate(revised_queries):
    response = llm.invoke(ques)
    res_list.append(response)
    print(f"完成 {i+1} 个回答")


In [11]:
for i, res in enumerate(res_list):
    print(f"查询 {i+1} 的答案是：{res.content}")

查询 1 的答案是：Calix[4]pyrrole is a type of macrocyclic compound that consists of four pyrrole subunits linked by methylene bridges at the α-positions of the pyrrole rings. This structure forms a cyclic tetramer, creating a bowl-like conformation. Calix[4]pyrrole is known for its ability to act as a host molecule in supramolecular chemistry, particularly for anion binding. The electron-rich pyrrole rings can interact with anions through hydrogen bonding and other non-covalent interactions, making calix[4]pyrrole an effective anion receptor. Its binding properties can be modified by introducing various substituents on the pyrrole rings or the methylene bridges, allowing for the design of receptors with specific selectivity and affinity for different anions. Calix[4]pyrrole and its derivatives have potential applications in areas such as sensing, separation, and catalysis.
查询 2 的答案是：The pKa value of methanesulfonic acid (MSA) is approximately -1.9. This indicates that it is a strong acid, as 

**Note on Streaming with QueryManager:**

The standard `QueryManager.submit_query()` returns a `Future` that resolves to the *final* state of the LangGraph execution. It doesn't inherently provide access to the intermediate states generated by each node.

To observe the step-by-step execution and intermediate state changes, you would typically need to interact directly with the LangGraph instance using its `stream()` method, as demonstrated in Part 2 below. Modifying the `QueryManager` to expose this stream would require significant changes to its asynchronous task handling and result reporting.

## Part 2: Direct Execution via Graph Stream

In [13]:
if not llm:
    print("Skipping Direct Graph Stream test due to LLM initialization failure.")
else:
    print("\n--- Starting Direct Graph Stream Test ---")

    # 1. Create graph instance
    print("Creating graph instance...")
    graph = create_query_graph()
    print("Graph instance created.")

    # 2. Manually create initial state dictionary
    print("Creating initial state...")
    # Use the same query as Part 1 for comparison
    # query_text_stream = "What proteins does DrugA bind to?"

    for query_text_stream in [queries[4], queries[6], queries[9]]:

        try:
            # Ensure we get a list of strings
            available_classes_stream = sorted([cls.name for cls in test_onto.classes() if isinstance(cls, ThingClass)])
            available_data_props_stream = sorted([dp.name for dp in test_onto.data_properties() if isinstance(dp, DataPropertyClass)])
            available_object_props_stream = sorted([op.name for op in test_onto.object_properties() if isinstance(op, ObjectPropertyClass)])
        except Exception as e:
            print(f"Error getting class names: {e}")
            available_classes_stream = []

        initial_state = {
            "query": query_text_stream,
            "source_ontology": test_ontology_settings, # Pass the actual ontology object
            "available_classes": available_classes_stream,
            "available_data_properties": available_data_props_stream,
            "available_object_properties": available_object_props_stream,
            "query_type": "information_retrieval",
            "query_strategy": None,
            "originating_team": "test_notebook_stream",
            "originating_stage": "manual_stream_test",
            "query_results": {},
            "normalized_query": None,
            "execution_plan": None,
            "validation_report": None,
            "sparql_query": None,
            "status": "initialized",
            "stage": "initialized",
            "previous_stage": None,
            "error": None,
            "messages": [] # LangGraph expects messages field
        }
        print("Initial state prepared.")
        # print(json.dumps(initial_state, indent=2, default=str)) # Optionally print initial state (ontology won't serialize well)

        # 3. Execute and iterate stream
        print("\n--- Streaming Graph Execution --- ")
        try:
            stream_counter = 0
            # Use stream method to get intermediate steps
            for chunk in graph.stream(initial_state):
                stream_counter += 1
                print(f"\n--- Chunk {stream_counter} --- ")
                # Chunks are dictionaries where keys are node names that just ran
                # and values are the outputs (state updates) returned by that node
                # Use default=str to handle potential non-serializable objects in the state
                print(json.dumps(chunk, indent=2, default=str))
                print("-" * 30)
            print("\n--- Graph Stream Finished --- ")
        except Exception as e:
            print(f"\nError during graph stream: {e}")
            import traceback
            traceback.print_exc() # Print full traceback for stream errors
        
        print(f"query:{query_text_stream} has been finished.")

    print("--- Direct Graph Stream Test Finished ---")


--- Starting Direct Graph Stream Test ---
Creating graph instance...
Graph instance created.
Creating initial state...
Initial state prepared.

--- Streaming Graph Execution --- 
[('system', 'You are an expert ontology query parser. Your task is to convert natural language queries into a structured format.\n1. Strictly adhere to the NormalizedQuery JSON schema for the output.\n2. Refer to the provided list of available ontology classes to identify entities.\n3. Refer to the provided lists of data properties and object properties to identify property relationships.\n4. Note that there are SourcedInformation objects that provide additional metadata. When queries involve concepts like "source", "description", or "definition", consider that these information are not related to relations.'), ('user', 'Available classes: 1,3,5-triethyl-2,4,6-trimethylamine, 1,3-diynyl, 1,4-triazole, 1:1_inclusion_complexes, 1H_NMR_spectroscopic_titration, 1H_NMR_spectroscopic_titrations, 1H_NMR_spectrum, 1_

In [8]:
print(initial_state["normalized_query"])

None


# QueryManager 检查


In [7]:
import threading
import traceback
import sys

def check_threads():
    """检查当前进程中的活跃线程"""
    print(f"当前活跃线程数: {threading.active_count()}")
    
    print("\n当前活跃线程:")
    for t in threading.enumerate():
        print(f"- {t.name} (daemon: {t.daemon}, 活动: {t.is_alive()})")
    
    print("\n线程调用栈:")
    query_manager_threads = []
    for thread_id, frame in sys._current_frames().items():
        thread_name = "Unknown"
        for t in threading.enumerate():
            if t.ident == thread_id:
                thread_name = t.name
                break
        
        # 检查是否是QueryManager相关线程
        is_query_thread = False
        stack_trace = traceback.extract_stack(frame)
        for filename, _, _, _ in stack_trace:
            if "query_manager" in filename or "ThreadPool" in filename:
                is_query_thread = True
                query_manager_threads.append(thread_name)
                break
        
        print(f"线程ID: {thread_id}, 名称: {thread_name}{' (QueryManager相关)' if is_query_thread else ''}")
        for filename, lineno, name, line in stack_trace[-10:]:  # 只显示最近10个调用
            print(f"  文件: {filename.split('/')[-1]}, 行: {lineno}, 函数: {name}")
            if line:
                print(f"    代码: {line}")
        print("")
    
    if query_manager_threads:
        print(f"\n发现 {len(query_manager_threads)} 个QueryManager相关线程: {', '.join(query_manager_threads)}")
    else:
        print("\n未发现QueryManager相关线程")

# 执行检查
check_threads()

当前活跃线程数: 6

当前活跃线程:
- MainThread (daemon: False, 活动: True)
- IOPub (daemon: True, 活动: True)
- Heartbeat (daemon: True, 活动: True)
- Control (daemon: True, 活动: True)
- IPythonHistorySavingThread (daemon: True, 活动: True)
- Thread-1 (daemon: True, 活动: True)

线程调用栈:
线程ID: 13768, 名称: Thread-1
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1012, 函数: _bootstrap
    代码: self._bootstrap_inner()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1041, 函数: _bootstrap_inner
    代码: self.run()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\site-packages\ipykernel\parentpoller.py, 行: 93, 函数: run
    代码: result = ctypes.windll.kernel32.WaitForMultipleObjects(  # type:ignore[attr-defined]

线程ID: 33716, 名称: IPythonHistorySavingThread
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1012, 函数: _bootstrap
    代码: self._bootstrap_inner()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1041, 函数: _bootstrap_inner
    代码: sel

In [8]:
# 查看缓存内容（如果有缓存的查询）
cache_content = query_manager.query_queue_manager.cache.cache
print(f"缓存中的查询数量: {len(cache_content)}")

# 查看缓存的时间戳信息
timestamps = query_manager.query_queue_manager.cache.timestamps
if timestamps:
    print("\n缓存时间戳:")
    for key, timestamp in timestamps.items():
        print(f"查询: {key[:50]}... - 时间: {timestamp}")
        # 计算剩余有效时间
        ttl = query_manager.query_queue_manager.cache.ttl  # 默认3600秒（1小时）
        from datetime import datetime, timedelta
        remaining = timestamp + timedelta(seconds=ttl) - datetime.now()
        print(f"  剩余有效时间: {remaining}")

# 如果需要手动清除缓存
# query_manager.query_queue_manager.cache.clear()
# print("缓存已清除")

缓存中的查询数量: 0
